# Automated Marine Debris Detection using Deep Learning

## 1. Introduction and Problem Statement

## 2. Data Sources

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

### 3.2 Class Distribution

### 3.3 Image and Annotation Visualisation

### 3.4 Preprocessing and Data Splitting

In [2]:
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict
from help_funciton.evaluation_protocol import audit_splits

OUTPUT_DIR = Path("instance_version")
SPLIT_SEED = 42
REGENERATE_SPLIT = False  # Changing this requires retraining every model.
split_paths = [OUTPUT_DIR / "instances_train.json", OUTPUT_DIR / "instances_test.json"]
exists = [p.exists() for p in split_paths]
if any(exists) and not all(exists) and not REGENERATE_SPLIT:
    raise RuntimeError("Only one split file exists. Restore both committed split files.")

if REGENERATE_SPLIT or not all(exists):
    with (OUTPUT_DIR / "instances_train_trashcan.json").open(encoding="utf-8") as f:
        data = json.load(f)
    video_groups = defaultdict(list)
    image_categories = defaultdict(set)
    for image in data["images"]:
        video_groups[image["file_name"].split("_frame")[0]].append(image)
    for ann in data["annotations"]:
        image_categories[ann["image_id"]].add(ann["category_id"])
    all_categories = {c["id"] for c in data["categories"]}
    rng = random.Random(SPLIT_SEED)
    for attempt in range(200):
        video_ids = sorted(video_groups)
        rng.shuffle(video_ids)
        selected, count = set(), 0
        for video_id in video_ids:
            if count >= 1000:
                break
            selected.add(video_id)
            count += len(video_groups[video_id])
        test_images = [im for video, group in video_groups.items() if video in selected for im in group]
        train_images = [im for video, group in video_groups.items() if video not in selected for im in group]
        train_cats = set().union(*(image_categories[im["id"]] for im in train_images))
        test_cats = set().union(*(image_categories[im["id"]] for im in test_images))
        if train_cats == test_cats == all_categories:
            break
    else:
        raise RuntimeError("Failed to find a video-disjoint split with all categories.")
    for output_path, images in zip(split_paths, [train_images, test_images]):
        image_ids = {im["id"] for im in images}
        split = {"info": data.get("info", {}), "licenses": data.get("licenses", []),
                 "images": images, "categories": data["categories"],
                 "annotations": [a for a in data["annotations"] if a["image_id"] in image_ids]}
        output_path.write_text(json.dumps(split), encoding="utf-8")
else:
    print("Reusing the committed train/test split; no annotations were overwritten.")

split_data = [json.loads(p.read_text(encoding="utf-8")) for p in split_paths]
validation_data = json.loads((OUTPUT_DIR / "instances_val_trashcan.json").read_text(encoding="utf-8"))
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r"^Validation and (train|test) share .*", category=UserWarning)
    split_audit = audit_splits(split_data[0], split_data[1], validation_data)
print(json.dumps({key: value for key, value in split_audit.items() if key != "warnings"}, indent=2))

Reusing the committed train/test split; no annotations were overwritten.
{
  "splits": {
    "train": {
      "images": 4970,
      "annotations": 7772,
      "videos": 259,
      "empty_images": 126
    },
    "test": {
      "images": 1095,
      "annotations": 1768,
      "videos": 51,
      "empty_images": 3
    },
    "validation": {
      "images": 1147,
      "annotations": 2588,
      "videos": 115,
      "empty_images": 0
    }
  },
  "overlap": {
    "validation_train": {
      "filenames": 0,
      "videos": 97,
      "validation_images": 953
    },
    "validation_test": {
      "filenames": 0,
      "videos": 16,
      "validation_images": 160
    }
  }
}


## 4. Models and Methods

### 4.1 Faster R-CNN

In [3]:
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


def create_faster_rcnn_model(num_classes, pretrained=True):
    model = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None,
        weights_backbone=None,
    )
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, num_classes)
    return model


### 4.2 Mask R-CNN

In [4]:
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor
)
from torchvision.models.detection.mask_rcnn import (
    MaskRCNNPredictor
)

def create_model(num_classes, pretrained=True):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None,
        weights_backbone=None
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        hidden_layer,
        num_classes
    )
    return model

Baseline

In [5]:
def load_baseline_model(device):
    model = maskrcnn_resnet50_fpn_v2(
            weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
        )
    model.to(device=device)
    model.eval()

    return model

## 5. Experimental Setup

### 5.1 Training Configuration

In [6]:
import torch
from pycocotools.coco import COCO
from help_funciton.marine_dataset import MarineDataset

Transformer

In [7]:
from torchvision.transforms import v2 as T
def get_transform(train):
    transforms = []
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    transforms.append(T.ToDtype(torch.float, scale=True))
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)

Dataset setup

In [8]:
from help_funciton.engine import train_one_epoch, evaluate
from help_funciton.checkpoints import save_model_checkpoint, load_model_checkpoint, file_sha256, validate_checkpoint_metadata
import numpy as np

TRAINING_SEED = 42


def seed_training():
    random.seed(TRAINING_SEED)
    np.random.seed(TRAINING_SEED)
    torch.manual_seed(TRAINING_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(TRAINING_SEED)


trained_model_names = set()
trained_metadata = {}
checkpoint_annotation_files = {"train": split_paths[0], "test": split_paths[1]}


def finish_training(model, name, checkpoint_path, architecture, optimizer_name, epochs, flip=False):
    metadata = save_model_checkpoint(
        model, checkpoint_path, architecture=architecture, category_ids=category_id,
        annotation_files=checkpoint_annotation_files,
        training_config={"seed": TRAINING_SEED, "epochs": epochs,
                         "optimizer": optimizer_name, "horizontal_flip": flip,
                         "validation_image_ids": sorted(im["id"] for im in validation_data["images"]),
                         "validation_annotation_sha256": file_sha256(annotation_file_val)},
    )
    trained_model_names.add(name)
    trained_metadata[name] = metadata
    model.to("cpu")
    print(f"Saved {name} evaluation checkpoint to {checkpoint_path}")

# train on the accelerator or on the CPU, if an accelerator is not available
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")

# One shared foreground label mapping for train, validation and test.
image_dir = "instance_version/train"
annotation_file = "instance_version/instances_train.json"
category_id = COCO(annotation_file).getCatIds()
num_classes = len(category_id) + 1
dataset = MarineDataset(image_dir, annotation_file, category_id, transforms=get_transform(train=False))
image_dir_val = "instance_version/val"
annotation_file_val = "instance_version/instances_val_trashcan.json"
category_id_val = category_id
num_classes_val = len(category_id_val) + 1
dataset_val = MarineDataset(image_dir_val, annotation_file_val, category_id_val, transforms=get_transform(train=False))

dataset_trans = MarineDataset(image_dir, annotation_file, category_id, transforms=get_transform(train=True))


# define training and validation data loaders
def collate_fn(batch):
    return tuple(zip(*batch))
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

data_loader_val = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

data_loader_trans = torch.utils.data.DataLoader(
    dataset_trans,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)


In [9]:
seed_training()
trained_model_names.discard("Faster R-CNN / SGD")
faster_model = create_faster_rcnn_model(num_classes).to(device)
faster_optimizer = torch.optim.SGD(
    [p for p in faster_model.parameters() if p.requires_grad],
    lr=0.005, momentum=0.9, weight_decay=0.0005,
)
faster_scheduler = torch.optim.lr_scheduler.StepLR(faster_optimizer, step_size=3, gamma=0.1)
faster_num_epochs = 15
for epoch in range(faster_num_epochs):
    train_one_epoch(faster_model, faster_optimizer, data_loader, device, epoch, print_freq=10)
    faster_scheduler.step()
    evaluate(faster_model, data_loader_val, device=device, iou_types=["bbox"])

finish_training(faster_model, "Faster R-CNN / SGD", "checkpoints/faster_rcnn_sgd.pth",
                "fasterrcnn_resnet50_fpn_v2", "SGD + StepLR", faster_num_epochs)
del faster_optimizer, faster_scheduler


Train sgd+steplr

In [10]:
seed_training()
trained_model_names.discard('Mask R-CNN / SGD')
model = create_model(num_classes)

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    weight_decay=0.0005,
    momentum=0.9
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

num_epochs = 15

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # Monitor validation only; reserve test metrics for final reporting.
    evaluate(model, data_loader_val, device=device)
finish_training(model, 'Mask R-CNN / SGD', "checkpoints/mask_rcnn_sgd.pth",
                "maskrcnn_resnet50_fpn_v2", 'SGD + StepLR', num_epochs, flip=False)
del optimizer, lr_scheduler


Train adamw+cosineAnnealingLR

In [11]:
seed_training()
trained_model_names.discard('Mask R-CNN / AdamW')
model_adamw = create_model(num_classes)

model_adamw.to(device)

# construct an optimizer
params = [p for p in model_adamw.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    params,
     lr=0.0001,
    weight_decay=0.0001
)

num_epochs = 15

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)



for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model_adamw, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # Monitor validation only; reserve test metrics for final reporting.
    evaluate(model_adamw, data_loader_val, device=device)
finish_training(model_adamw, 'Mask R-CNN / AdamW', "checkpoints/mask_rcnn_adamw.pth",
                "maskrcnn_resnet50_fpn_v2", 'AdamW + CosineAnnealingLR', num_epochs, flip=False)
del optimizer, lr_scheduler


train sgd+stepLR using trans data

In [2]:
seed_training()
trained_model_names.discard('Mask R-CNN / SGD + flip')
# get the model using our helper function
model_sgd_trans = create_model(num_classes)

# move model to the right device
model_sgd_trans.to(device)

# construct an optimizer
params = [p for p in model_sgd_trans.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    weight_decay=0.0005,
    momentum=0.9
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

num_epochs = 15

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model_sgd_trans, optimizer, data_loader_trans, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # Monitor validation only; reserve test metrics for final reporting.
    evaluate(model_sgd_trans, data_loader_val, device=device)
finish_training(model_sgd_trans, 'Mask R-CNN / SGD + flip', "checkpoints/mask_rcnn_sgd_flip.pth",
                "maskrcnn_resnet50_fpn_v2", 'SGD + StepLR', num_epochs, flip=True)
del optimizer, lr_scheduler


Test dataset setup

In [3]:
image_dir_test = "instance_version/train"
annotation_file_test = "instance_version/instances_test.json"
category_id_test = category_id
num_classes_test = len(category_id_test) + 1
dataset_test = MarineDataset(image_dir_test, annotation_file_test, category_id_test, transforms=get_transform(train=False))


data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

### 5.2 Evaluation Metrics

In [4]:
from IPython.display import display
import torchvision
import pandas as pd
import matplotlib.pyplot as plt
from help_funciton.evaluation import (
    summarize_evaluators, per_class_metrics, plot_precision_recall,
    plot_predictions, plot_model_comparison, export_evaluation_results,
)

MODEL_CHECKPOINTS = {}
model_specs = {
    "Faster R-CNN / SGD": {
        "variable": "faster_model", "factory": create_faster_rcnn_model,
        "architecture": "fasterrcnn_resnet50_fpn_v2", "iou_types": ["bbox"],
        "checkpoint": "checkpoints/faster_rcnn_sgd.pth",
    },
    "Mask R-CNN / SGD": {
        "variable": "model", "factory": create_model,
        "architecture": "maskrcnn_resnet50_fpn_v2", "iou_types": ["bbox", "segm"],
        "checkpoint": "checkpoints/mask_rcnn_sgd.pth",
    },
    "Mask R-CNN / AdamW": {
        "variable": "model_adamw", "factory": create_model,
        "architecture": "maskrcnn_resnet50_fpn_v2", "iou_types": ["bbox", "segm"],
        "checkpoint": "checkpoints/mask_rcnn_adamw.pth",
    },
    "Mask R-CNN / SGD + flip": {
        "variable": "model_sgd_trans", "factory": create_model,
        "architecture": "maskrcnn_resnet50_fpn_v2", "iou_types": ["bbox", "segm"],
        "checkpoint": "checkpoints/mask_rcnn_sgd_flip.pth",
    },
}
if set(MODEL_CHECKPOINTS) - set(model_specs):
    raise ValueError("Unknown checkpoint name; use a key from model_specs.")
models_to_evaluate, checkpoint_sources = {}, {}
for name, spec in model_specs.items():
    use_memory = name not in MODEL_CHECKPOINTS and name in trained_model_names
    checkpoint_path = Path(MODEL_CHECKPOINTS.get(name, spec["checkpoint"]))
    if use_memory:
        validate_checkpoint_metadata(
            trained_metadata[name], architecture=spec["architecture"], category_ids=category_id,
            annotation_files=checkpoint_annotation_files,
        )
        candidate = globals()[spec["variable"]]
        checkpoint_sources[name] = {"source": "completed_training_in_this_kernel",
                                    "verification": "verified", "metadata": trained_metadata[name]}
    elif name in MODEL_CHECKPOINTS or checkpoint_path.is_file():
        candidate = spec["factory"](num_classes, pretrained=False)
        provenance = load_model_checkpoint(
            candidate, checkpoint_path, architecture=spec["architecture"], category_ids=category_id,
            annotation_files=checkpoint_annotation_files,
        )
        checkpoint_sources[name] = {"source": str(checkpoint_path),
                                    "sha256": file_sha256(checkpoint_path), **provenance}
    else:
        print(f"Skipping {name}: no completed training or checkpoint ({checkpoint_path}).")
        continue
    models_to_evaluate[name] = candidate.to("cpu")
if not models_to_evaluate:
    raise RuntimeError("Train at least one model in section 5.1 or provide MODEL_CHECKPOINTS.")

class_names = {dataset.cat_to_label[c["id"]]: c["name"] for c in dataset.coco.dataset["categories"]}
assert dataset.cat_to_label == dataset_val.cat_to_label == dataset_test.cat_to_label
evaluators = {}
for name, candidate in models_to_evaluate.items():
    print(f"\nEvaluating {name} on {len(dataset_test)} held-out images")
    try:
        candidate.to(device)
        evaluators[name] = evaluate(candidate, data_loader_test, device=device,
                                    iou_types=model_specs[name]["iou_types"])
    finally:
        candidate.to("cpu")
summary = summarize_evaluators(evaluators)
display(summary.round(4))


,model,iou_type,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR1,AR10,AR100,AR_small,AR_medium,AR_large
0,Faster R-CNN / SGD,bbox,0.0845,0.1279,0.0912,0.1215,0.0939,0.1730,0.1627,0.1903,0.1903,0.1555,0.2455,0.3449
1,Mask R-CNN / SGD,bbox,0.0970,0.1575,0.1016,0.1171,0.1338,0.1250,0.1862,0.2143,0.2143,0.1640,0.2554,0.3128
2,Mask R-CNN / SGD,segm,0.0874,0.1489,0.0901,0.0901,0.1056,0.1592,0.1620,0.1798,0.1798,0.1449,0.2175,0.2574
3,Mask R-CNN / AdamW,bbox,0.0730,0.1166,0.0750,0.1045,0.0942,0.1398,0.1207,0.1378,0.1378,0.1378,0.1635,0.2739
4,Mask R-CNN / AdamW,segm,0.0666,0.1160,0.0654,0.0791,0.0858,0.1683,0.1141,0.1242,0.1242,0.1184,0.1609,0.2281
5,Mask R-CNN / SGD + flip,bbox,0.0931,0.1449,0.1004,0.1281,0.1224,0.1428,0.1981,0.2440,0.2440,0.1889,0.2829,0.3350
6,Mask R-CNN / SGD + flip,segm,0.0787,0.1351,0.0790,0.0971,0.0929,0.1557,0.1626,0.1910,0.1910,0.1644,0.2217,0.2704


In [5]:
OUTPUT_DIR_EVAL = Path("evaluation_results") / "held_out_test"
report = export_evaluation_results(evaluators, OUTPUT_DIR_EVAL, class_names)
display(report["summary"].round(4))
display(report["per_class"].round(4))

annotation_paths = [annotation_file, annotation_file_val, annotation_file_test]
metadata = {
    "split": "held_out_test", "split_audit": split_audit,
    "category_ids_in_training_order": list(category_id),
    "class_names": class_names, "models": list(models_to_evaluate),
    "checkpoint_sources": checkpoint_sources,
    "torch_version": str(torch.__version__), "torchvision_version": str(torchvision.__version__),
    "annotation_sha256": {str(p): file_sha256(p) for p in annotation_paths},
    "evaluated_image_ids": {name: list(map(int, evaluator.coco_eval["bbox"].params.imgIds))
                            for name, evaluator in evaluators.items()},
    "inference_settings": {name: {
        "roi_heads": {key: getattr(candidate.roi_heads, key) for key in
                      ("score_thresh", "nms_thresh", "detections_per_img")},
        "min_size": list(candidate.transform.min_size), "max_size": candidate.transform.max_size,
        "iou_types": model_specs[name]["iou_types"],
    } for name, candidate in models_to_evaluate.items()},
    "extra_metric_score_threshold": None, "mask_threshold": 0.5,
    "device": str(device), "training_seed": TRAINING_SEED,
}
(OUTPUT_DIR_EVAL / "evaluation_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved tables, PR curves, comparison plot and evaluation metadata to {OUTPUT_DIR_EVAL.resolve()}")


,model,iou_type,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR1,AR10,AR100,AR_small,AR_medium,AR_large
0,Faster R-CNN / SGD,bbox,0.0845,0.1279,0.0912,0.1215,0.0939,0.1730,0.1627,0.1903,0.1903,0.1555,0.2455,0.3449
1,Mask R-CNN / SGD,bbox,0.0970,0.1575,0.1016,0.1171,0.1338,0.1250,0.1862,0.2143,0.2143,0.1640,0.2554,0.3128
2,Mask R-CNN / SGD,segm,0.0874,0.1489,0.0901,0.0901,0.1056,0.1592,0.1620,0.1798,0.1798,0.1449,0.2175,0.2574
3,Mask R-CNN / AdamW,bbox,0.0730,0.1166,0.0750,0.1045,0.0942,0.1398,0.1207,0.1378,0.1378,0.1378,0.1635,0.2739
4,Mask R-CNN / AdamW,segm,0.0666,0.1160,0.0654,0.0791,0.0858,0.1683,0.1141,0.1242,0.1242,0.1184,0.1609,0.2281
5,Mask R-CNN / SGD + flip,bbox,0.0931,0.1449,0.1004,0.1281,0.1224,0.1428,0.1981,0.2440,0.2440,0.1889,0.2829,0.3350
6,Mask R-CNN / SGD + flip,segm,0.0787,0.1351,0.0790,0.0971,0.0929,0.1557,0.1626,0.1910,0.1910,0.1644,0.2217,0.2704


,model,iou_type,category_id,class_name,gt_instances,gt_images,AP,AP50,AR100
0,Faster R-CNN / SGD,bbox,1,rov,364,310,0.1883,0.2820,0.4970
1,Faster R-CNN / SGD,bbox,2,plant,11,7,0.0004,0.0006,0.0636
2,Faster R-CNN / SGD,bbox,3,animal_fish,151,149,0.3096,0.4836,0.5483
3,Faster R-CNN / SGD,bbox,4,animal_starfish,62,17,0.0025,0.0067,0.0435
4,Faster R-CNN / SGD,bbox,5,animal_shells,34,26,0.0000,0.0000,0.0000
...,...,...,...,...,...,...,...,...,...
149,Mask R-CNN / SGD + flip,segm,18,trash_branch,21,21,0.3058,0.3911,0.7381
150,Mask R-CNN / SGD + flip,segm,19,trash_wreckage,21,21,0.0000,0.0000,0.0000
151,Mask R-CNN / SGD + flip,segm,20,trash_tarp,3,3,0.0000,0.0000,0.0000
152,Mask R-CNN / SGD + flip,segm,21,trash_rope,30,30,0.0005,0.0048,0.0167


In [6]:
import numpy as np

sample_indices = np.random.default_rng(42).choice(len(dataset_test), min(3, len(dataset_test)), replace=False).tolist()
for index, (name, candidate) in enumerate(models_to_evaluate.items(), start=1):
    try:
        candidate.to(device)
        fig = plot_predictions(candidate, dataset_test, device, class_names,
                               indices=sample_indices, score_threshold=0.5)
    finally:
        candidate.to("cpu")
    try:
        fig.suptitle(name, y=1.01)
        fig.savefig(OUTPUT_DIR_EVAL / f"{index:02d}_qualitative.png", dpi=160, bbox_inches="tight")
        display(fig)
    finally:
        plt.close(fig)


In [7]:
bbox_comparison = report["summary"].query("iou_type == 'bbox'").set_index("model")
display(bbox_comparison[["mAP", "AP50", "AP75", "AR100"]].round(4))
segm_comparison = report["summary"].query("iou_type == 'segm'").set_index("model")
if not segm_comparison.empty:
    display(segm_comparison[["mAP", "AP50", "AP75", "AR100"]].round(4))
fig = plot_model_comparison(report["summary"])
display(fig)
plt.close(fig)


,mAP,AP50,AP75,AR100
model,,,,
Faster R-CNN / SGD,0.0845,0.1279,0.0912,0.1903
Mask R-CNN / SGD,0.0970,0.1575,0.1016,0.2143
Mask R-CNN / AdamW,0.0730,0.1166,0.0750,0.1378
Mask R-CNN / SGD + flip,0.0931,0.1449,0.1004,0.2440


,mAP,AP50,AP75,AR100
model,,,,
Mask R-CNN / SGD,0.0874,0.1489,0.0901,0.1798
Mask R-CNN / AdamW,0.0666,0.1160,0.0654,0.1242
Mask R-CNN / SGD + flip,0.0787,0.1351,0.0790,0.1910


## 6. Results

### 6.1 Quantitative Results

### 6.2 Qualitative Results

### 6.3 Model Comparison

## 7. Conclusion and Future Work

## References